In [1]:
%pip install torch pandas numpy scikit-learn

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.2.1 -> 26.2.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import hashlib
import pickle
import random
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from sklearn.metrics import (
    accuracy_score,
    brier_score_loss,
    confusion_matrix,
    roc_auc_score,
)
from sklearn.model_selection import train_test_split
from torch import nn
from torch.nn import functional as F
from torch.nn.utils.rnn import (
    pack_padded_sequence,
    pad_packed_sequence,
)
from torch.utils.data import DataLoader

DATA_DIR = Path(r"C:\Users\jhadz\Downloads\popbob")
SEED = 43
BATCH_SIZE = 32
MAX_EPOCHS = 20
PATIENCE = 4
LEARNING_RATE = 0.0003
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seed(SEED)
print("Using device:", DEVICE)

# This is the v1 model saved by your first notebook.
with open(DATA_DIR / "dkt_model.pkl", "rb") as file:
    v1_bundle = pickle.load(file)  # Load only your trusted local pickle.

SKILLS = v1_bundle["metadata"]["skills"]
K = len(SKILLS)
HIDDEN = v1_bundle["metadata"]["hidden"]
SKILL_TO_IDX = {skill: i for i, skill in enumerate(SKILLS)}

assert SKILLS == [f"S{i}" for i in range(1, 8)]
assert v1_bundle["metadata"]["input_size"] == 2 * K

v1_weights = {
    name: torch.from_numpy(value.copy())
    for name, value in v1_bundle["state_dict"].items()
}

print("Loaded v1 model with skills:", SKILLS)

def read_data(path):
    df = pd.read_csv(
        path,
        dtype={
            "user_id": str,
            "skill_code": str,
            "time_done": "int64",
            "y": "int64",
        },
    )

    assert set(df.columns) == {
        "user_id", "skill_code", "time_done", "y"
    }, f"Unexpected columns in {path}"
    assert df.notna().all().all(), f"Missing values in {path}"
    assert df["skill_code"].isin(SKILLS).all(), f"Unknown skill in {path}"
    assert df["y"].isin([0, 1]).all(), f"Invalid y in {path}"

    return df.sort_values(
        ["user_id", "time_done"],
        kind="mergesort",
    ).reset_index(drop=True)

def make_sequences(df):
    sequences = []

    for _, group in df.groupby("user_id", sort=False):
        skills = torch.tensor(
            group["skill_code"].map(SKILL_TO_IDX).to_numpy(),
            dtype=torch.long,
        )
        answers = torch.tensor(
            group["y"].to_numpy(),
            dtype=torch.long,
        )
        sequences.append((skills, answers))

    return sequences

def collate_sequences(batch):
    lengths = torch.tensor([len(skills) for skills, _ in batch])
    max_length = int(lengths.max())
    batch_size = len(batch)

    x = torch.zeros(batch_size, max_length, 2 * K)
    target_skills = torch.zeros(
        batch_size, max_length, dtype=torch.long
    )
    target_y = torch.zeros(batch_size, max_length)
    mask = torch.arange(max_length)[None, :] < lengths[:, None]

    for i, (skills, answers) in enumerate(batch):
        n = len(skills)
        target_skills[i, :n] = skills
        target_y[i, :n] = answers.float()

        # Input at t uses the PREVIOUS answer.
        # The first answer has an all-zero input.
        if n > 1:
            previous = skills[:-1] + K * answers[:-1]
            x[i, torch.arange(1, n), previous] = 1.0

    return x, lengths, target_skills, target_y, mask

def make_loader(sequences, shuffle):
    return DataLoader(
        sequences,
        batch_size=BATCH_SIZE,
        shuffle=shuffle,
        collate_fn=collate_sequences,
    )

class DKT(nn.Module):
    def __init__(self, k=7, hidden=100):
        super().__init__()
        self.lstm = nn.LSTM(
            2 * k,
            hidden,
            batch_first=True,
        )
        self.output = nn.Linear(hidden, k)

    def forward(self, x, lengths):
        packed = pack_padded_sequence(
            x,
            lengths.cpu(),
            batch_first=True,
            enforce_sorted=False,
        )
        packed_output, _ = self.lstm(packed)
        output, _ = pad_packed_sequence(
            packed_output,
            batch_first=True,
            total_length=x.size(1),
        )
        return self.output(output)

def new_model_from_v1():
    model = DKT(K, HIDDEN)
    model.load_state_dict(v1_weights)
    return model.to(DEVICE)

def run_epoch(model, loader, optimizer=None):
    training = optimizer is not None
    model.train(training)

    total_loss = 0.0
    total_answers = 0

    for x, lengths, target_skills, target_y, mask in loader:
        x = x.to(DEVICE)
        target_skills = target_skills.to(DEVICE)
        target_y = target_y.to(DEVICE)
        mask = mask.to(DEVICE)

        with torch.set_grad_enabled(training):
            all_logits = model(x, lengths)
            chosen_logits = all_logits.gather(
                2,
                target_skills.unsqueeze(-1),
            ).squeeze(-1)

            answer_losses = F.binary_cross_entropy_with_logits(
                chosen_logits,
                target_y,
                reduction="none",
            )
            loss_sum = (answer_losses * mask).sum()
            loss = loss_sum / mask.sum()

            if training:
                optimizer.zero_grad()
                loss.backward()
                torch.nn.utils.clip_grad_norm_(
                    model.parameters(), 5.0
                )
                optimizer.step()

        total_loss += loss_sum.item()
        total_answers += mask.sum().item()

    return total_loss / total_answers

@torch.no_grad()
def collect_predictions(model, loader):
    model.eval()
    all_y, all_probability, all_skill = [], [], []

    for x, lengths, target_skills, target_y, mask in loader:
        logits = model(x.to(DEVICE), lengths)
        chosen_logits = logits.gather(
            2,
            target_skills.to(DEVICE).unsqueeze(-1),
        ).squeeze(-1)
        probability = torch.sigmoid(chosen_logits)

        mask = mask.to(DEVICE)
        all_y.append(target_y.to(DEVICE)[mask].cpu().numpy())
        all_probability.append(
            probability[mask].cpu().numpy()
        )
        all_skill.append(
            target_skills.to(DEVICE)[mask].cpu().numpy()
        )

    return (
        np.concatenate(all_y).astype(int),
        np.concatenate(all_probability),
        np.concatenate(all_skill),
    )

def metric_rows(y, probability, skill_idx, model_name):
    rows = []
    groups = [("overall", np.ones(len(y), dtype=bool))]
    groups += [
        (skill, skill_idx == i)
        for i, skill in enumerate(SKILLS)
    ]

    for name, selected in groups:
        actual = y[selected]
        predicted = probability[selected]

        if len(actual) == 0:
            continue

        rows.append({
            "model": model_name,
            "skill": name,
            "n_answers": len(actual),
            "correct_rate": actual.mean(),
            "accuracy_at_0.5": accuracy_score(
                actual, predicted >= 0.5
            ),
            "majority_baseline_accuracy": max(
                actual.mean(), 1 - actual.mean()
            ),
            "auc": (
                roc_auc_score(actual, predicted)
                if len(np.unique(actual)) == 2
                else np.nan
            ),
            "brier": brier_score_loss(actual, predicted),
        })

    return pd.DataFrame(rows)

# Open trainv2.csv only. Leave testv2.csv untouched for now.
trainv2_df = read_data(DATA_DIR / "trainv2.csv")
student_ids = trainv2_df["user_id"].unique()

fit_ids, validation_ids = train_test_split(
    student_ids,
    test_size=0.10,
    random_state=SEED,
)

fit_df = trainv2_df[
    trainv2_df["user_id"].isin(fit_ids)
]
validation_df = trainv2_df[
    trainv2_df["user_id"].isin(validation_ids)
]

fit_loader = make_loader(
    make_sequences(fit_df),
    shuffle=True,
)
validation_loader = make_loader(
    make_sequences(validation_df),
    shuffle=False,
)

print("V2 training rows:", len(trainv2_df))
print("Fitting students:", len(fit_ids))
print("Validation students:", len(validation_ids))

Using device: cpu
Loaded v1 model with skills: ['S1', 'S2', 'S3', 'S4', 'S5', 'S6', 'S7']
V2 training rows: 300000
Fitting students: 6606
Validation students: 734


In [3]:
tuning_model = new_model_from_v1()
optimizer = torch.optim.Adam(
    tuning_model.parameters(),
    lr=LEARNING_RATE,
)

best_validation_loss = float("inf")
best_epoch = 0
epochs_without_improvement = 0

for epoch in range(1, MAX_EPOCHS + 1):
    train_loss = run_epoch(
        tuning_model,
        fit_loader,
        optimizer,
    )
    validation_loss = run_epoch(
        tuning_model,
        validation_loader,
    )

    print(
        f"Epoch {epoch:02d} | "
        f"train loss: {train_loss:.4f} | "
        f"validation loss: {validation_loss:.4f}"
    )

    if validation_loss < best_validation_loss - 1e-4:
        best_validation_loss = validation_loss
        best_epoch = epoch
        epochs_without_improvement = 0
    else:
        epochs_without_improvement += 1
        if epochs_without_improvement >= PATIENCE:
            print("Early stopping")
            break

assert best_epoch > 0
print("Selected additional epochs:", best_epoch)

# Start AGAIN from v1, this time using all trainv2 students.
set_seed(SEED)
final_v2_model = new_model_from_v1()
final_optimizer = torch.optim.Adam(
    final_v2_model.parameters(),
    lr=LEARNING_RATE,
)
full_v2_loader = make_loader(
    make_sequences(trainv2_df),
    shuffle=True,
)

for epoch in range(1, best_epoch + 1):
    loss = run_epoch(
        final_v2_model,
        full_v2_loader,
        final_optimizer,
    )
    print(
        f"Final v2 epoch {epoch:02d} | "
        f"loss: {loss:.4f}"
    )

print("V2 fine-tuning complete.")

Epoch 01 | train loss: 0.4405 | validation loss: 0.4300
Epoch 02 | train loss: 0.4396 | validation loss: 0.4292
Epoch 03 | train loss: 0.4390 | validation loss: 0.4289
Epoch 04 | train loss: 0.4386 | validation loss: 0.4282
Epoch 05 | train loss: 0.4385 | validation loss: 0.4294
Epoch 06 | train loss: 0.4383 | validation loss: 0.4281
Epoch 07 | train loss: 0.4382 | validation loss: 0.4277
Epoch 08 | train loss: 0.4377 | validation loss: 0.4277
Epoch 09 | train loss: 0.4378 | validation loss: 0.4281
Epoch 10 | train loss: 0.4376 | validation loss: 0.4279
Epoch 11 | train loss: 0.4374 | validation loss: 0.4273
Epoch 12 | train loss: 0.4375 | validation loss: 0.4279
Epoch 13 | train loss: 0.4372 | validation loss: 0.4272
Epoch 14 | train loss: 0.4370 | validation loss: 0.4276
Epoch 15 | train loss: 0.4369 | validation loss: 0.4274
Epoch 16 | train loss: 0.4368 | validation loss: 0.4269
Epoch 17 | train loss: 0.4369 | validation loss: 0.4272
Epoch 18 | train loss: 0.4366 | validation loss:

In [4]:
testv2_df = read_data(DATA_DIR / "testv2.csv")

assert set(trainv2_df["user_id"]).isdisjoint(
    set(testv2_df["user_id"])
), "A student appears in both v2 files!"

testv2_loader = make_loader(
    make_sequences(testv2_df),
    shuffle=False,
)

original_v1_model = new_model_from_v1()

v1_y, v1_probability, v1_skill = collect_predictions(
    original_v1_model,
    testv2_loader,
)
v2_y, v2_probability, v2_skill = collect_predictions(
    final_v2_model,
    testv2_loader,
)

assert len(v1_y) == len(testv2_df) == len(v2_y)
assert np.array_equal(v1_y, v2_y)
assert np.array_equal(v1_skill, v2_skill)

v1_results = metric_rows(
    v1_y, v1_probability, v1_skill, "v1"
)
v2_results = metric_rows(
    v2_y, v2_probability, v2_skill, "v2"
)
comparison = pd.concat(
    [v1_results, v2_results],
    ignore_index=True,
)

print("V1 AND V2 ON THE SAME TESTV2 STUDENTS")
print(comparison.round(4).to_string(index=False))

print("\nV2 confusion matrix at the 0.5 cutoff:")
print("Rows = actual 0/1; columns = predicted 0/1")
print(
    confusion_matrix(
        v2_y,
        (v2_probability >= 0.5).astype(int),
    )
)

V1 AND V2 ON THE SAME TESTV2 STUDENTS
model   skill  n_answers  correct_rate  accuracy_at_0.5  majority_baseline_accuracy    auc  brier
   v1 overall     100000        0.7951           0.8057                      0.7951 0.7371 0.1414
   v1      S1      18519        0.8678           0.8686                      0.8678 0.7279 0.1048
   v1      S2      48070        0.7773           0.7798                      0.7773 0.6943 0.1581
   v1      S3       2940        0.8915           0.8918                      0.8915 0.7094 0.0898
   v1      S4      10157        0.6478           0.7246                      0.6478 0.7380 0.1887
   v1      S5      10232        0.9169           0.9169                      0.9169 0.6858 0.0714
   v1      S6       7109        0.6976           0.7168                      0.6976 0.7061 0.1875
   v1      S7       2973        0.8513           0.8544                      0.8513 0.6535 0.1205
   v2 overall     100000        0.7951           0.8072                      0.7

In [5]:
def sha256(path):
    digest = hashlib.sha256()
    with open(path, "rb") as file:
        for chunk in iter(lambda: file.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()

state_cpu = {
    name: value.detach().cpu().clone()
    for name, value in final_v2_model.state_dict().items()
}

metadata_v2 = {
    "skills": SKILLS,
    "hidden": HIDDEN,
    "input_size": 2 * K,
    "base_model": "dkt_model.pkl",
    "base_model_sha256": sha256(DATA_DIR / "dkt_model.pkl"),
    "fine_tuning_data": "trainv2.csv",
    "trainv2_sha256": sha256(DATA_DIR / "trainv2.csv"),
    "selected_additional_epochs": best_epoch,
    "learning_rate": LEARNING_RATE,
    "seed": SEED,
    "label_rule": (
        "y=1 means correct on first attempt without hint"
    ),
    "first_answer_input": "all-zero 14-value vector",
}

pt_path = DATA_DIR / "dkt_v2.pt"
pkl_path = DATA_DIR / "dkt_model_v2.pkl"
metrics_path = DATA_DIR / "dkt_v2_test_metrics.csv"

torch.save(
    {
        "state_dict": state_cpu,
        "metadata": metadata_v2,
    },
    pt_path,
)

with open(pkl_path, "wb") as file:
    pickle.dump(
        {
            "state_dict": {
                name: value.numpy()
                for name, value in state_cpu.items()
            },
            "metadata": metadata_v2,
        },
        file,
        protocol=pickle.HIGHEST_PROTOCOL,
    )

comparison.to_csv(metrics_path, index=False)

print("Saved:", pt_path)
print("Saved:", pkl_path)
print("Saved:", metrics_path)

Saved: C:\Users\jhadz\Downloads\popbob\dkt_v2.pt
Saved: C:\Users\jhadz\Downloads\popbob\dkt_model_v2.pkl
Saved: C:\Users\jhadz\Downloads\popbob\dkt_v2_test_metrics.csv


In [6]:
with open(DATA_DIR / "dkt_model_v2.pkl", "rb") as file:
    saved_v2 = pickle.load(file)  # Your trusted local file.

loaded_v2_model = DKT(
    k=len(saved_v2["metadata"]["skills"]),
    hidden=saved_v2["metadata"]["hidden"],
)
loaded_v2_model.load_state_dict({
    name: torch.from_numpy(value)
    for name, value in saved_v2["state_dict"].items()
})
loaded_v2_model.eval()

def predict_next_v2(history):
    """
    history is a chronological list of (skill_code, y) pairs.
    Example: [("S1", 1), ("S2", 0)]
    """
    skills = saved_v2["metadata"]["skills"]
    skill_to_idx = {
        skill: i for i, skill in enumerate(skills)
    }
    k = len(skills)
    x = torch.zeros(1, len(history) + 1, 2 * k)

    for t, (skill, answer) in enumerate(history):
        if skill not in skill_to_idx or answer not in (0, 1):
            raise ValueError(
                f"Invalid history item: {(skill, answer)}"
            )
        x[
            0,
            t + 1,
            skill_to_idx[skill] + k * answer,
        ] = 1.0

    with torch.no_grad():
        logits = loaded_v2_model(
            x,
            torch.tensor([len(history) + 1]),
        )[0, -1]
        probabilities = torch.sigmoid(logits).tolist()

    return dict(zip(skills, probabilities))

print("New student:", predict_next_v2([]))
print(
    "After two answers:",
    predict_next_v2([("S1", 1), ("S2", 0)]),
)

New student: {'S1': 0.7090041041374207, 'S2': 0.5880699753761292, 'S3': 0.6803558468818665, 'S4': 0.5007537007331848, 'S5': 0.6895141005516052, 'S6': 0.47007492184638977, 'S7': 0.6526695489883423}
After two answers: {'S1': 0.8839541077613831, 'S2': 0.6020190715789795, 'S3': 0.8127347826957703, 'S4': 0.49847134947776794, 'S5': 0.802260160446167, 'S6': 0.46786385774612427, 'S7': 0.7646653652191162}


In [7]:
accuracy_report = pd.read_csv(
    DATA_DIR / "dkt_v2_test_metrics.csv"
)

for model_name in ["v1", "v2"]:
    overall = accuracy_report[
        (accuracy_report["model"] == model_name)
        & (accuracy_report["skill"] == "overall")
    ].iloc[0]

    print(f"{model_name.upper()} overall accuracy: "
          f"{overall['accuracy_at_0.5']:.2%}")
    print(f"{model_name.upper()} AUC: "
          f"{overall['auc']:.4f}")
    print(f"{model_name.upper()} Brier score: "
          f"{overall['brier']:.4f}")
    print()

print("V2 accuracy by skill:")
v2_skills = accuracy_report[
    (accuracy_report["model"] == "v2")
    & (accuracy_report["skill"] != "overall")
]

for _, row in v2_skills.iterrows():
    print(
        f"{row['skill']}: {row['accuracy_at_0.5']:.2%} "
        f"(majority baseline: "
        f"{row['majority_baseline_accuracy']:.2%})"
    )

V1 overall accuracy: 80.57%
V1 AUC: 0.7371
V1 Brier score: 0.1414

V2 overall accuracy: 80.72%
V2 AUC: 0.7455
V2 Brier score: 0.1400

V2 accuracy by skill:
S1: 86.88% (majority baseline: 86.78%)
S2: 78.18% (majority baseline: 77.73%)
S3: 89.32% (majority baseline: 89.15%)
S4: 72.34% (majority baseline: 64.78%)
S5: 91.69% (majority baseline: 91.69%)
S6: 72.43% (majority baseline: 69.76%)
S7: 85.47% (majority baseline: 85.13%)
